<a href="https://colab.research.google.com/github/romidj/PFE-CpG-islands/blob/main/CpG_epigenetics_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1 — Epigenetic Data Download


---


What this script does
---------------------
1. Defines the exact URL patterns for Roadmap Epigenomics bigWig files
2. Probes every (epigenome × mark) URL with HTTP HEAD — no data downloaded yet
3. Builds an availability matrix and saves it as mark_availability.csv
4. Downloads only the files that exist, with resume support and integrity checks
5. Reports a clean summary of what was and wasn't found

Key engineering decisions explained inline.

Directory structure produced
-----------------------------
PFE/data/epigenetics/
├── bigwig/
│   ├── E003-H3K4me3.pval.signal.bigwig
│   ├── E003-H3K4me1.pval.signal.bigwig
│   ├── ...
│   └── E003_WGBS_FractionalMethylation.bigwig
└── mark_availability.csv
"""

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Imports
# ─────────────────────────────────────────────────────────────────────────────

import os
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration — change only these values if your setup differs
# ─────────────────────────────────────────────────────────────────────────────

DRIVE_ROOT   = Path('/content/drive/MyDrive/PFE')
BIGWIG_DIR   = DRIVE_ROOT / 'data' / 'epigenetics' / 'bigwig'
PROC_DIR     = DRIVE_ROOT / 'data' / 'epigenetics' / 'processed'
AVAIL_PATH   = DRIVE_ROOT / 'data' / 'epigenetics' / 'mark_availability.csv'

In [ ]:
BIGWIG_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ── Epigenomes ────────────────────────────────────────────────────────────────
# These are the 10 Roadmap epigenomes selected for the project.
# Format: {EID: human-readable label}
EPIGENOMES = {
    'E003': 'H1_ESC',
    'E017': 'IMR90_fetal_lung',
    'E050': 'brain_angular_gyrus',
    'E067': 'brain_cortex',
    'E062': 'monocytes',
    'E045': 'HSC',
    'E065': 'aorta',
    'E071': 'brain_hippocampus',
    'E104': 'heart_right_atrium',
    'E116': 'GM12878_lymphoblastoid',
}


In [ ]:
# ── Epigenetic marks ──────────────────────────────────────────────────────────
# Split into two groups because they come from different Roadmap data sources
# and have completely different URL patterns and file formats.

HISTONE_MARKS = [
    'H3K4me3',    # active promoters — sharp peak at TSS
    'H3K4me1',    # primed/active enhancers — broad around enhancers
    'H3K36me3',   # transcribed gene bodies — broad
    'H3K27me3',   # Polycomb repression — broad
    'H3K9me3',    # constitutive heterochromatin — broad
    'H3K27ac',    # active enhancers/promoters — sharp
    'H3K9ac',     # active promoters — sharp
    'DNase',      # open chromatin / accessibility
]
# WGBS is handled separately because:
# (1) its URL pattern is completely different
# (2) its values are fractional methylation [0,1], not -log10(p-value)
# (3) it needs different clipping/normalization later
WGBS_MARK = 'WGBS'

ALL_MARKS = HISTONE_MARKS + [WGBS_MARK]   # 9 total channels in this order


In [ ]:
# ── Roadmap URL patterns ──────────────────────────────────────────────────────
# These are the EXACT canonical patterns from the Roadmap Epigenomics portal.
# Verified against https://egg2.wustl.edu/roadmap/data/

# Histone marks and DNase: p-value signal tracks from MACS2
# Files are named: {EID}-{MARK}.pval.signal.bigwig
# All uppercase mark names, DNase uses "DNase" not "DNASE"
BASE_HISTONE = (
    "https://egg2.wustl.edu/roadmap/data/byFileType/"
    "signal/consolidated/macs2signal/pval/"
)
# ── Roadmap URL patterns ──────────────────────────────────────────────────────
# These are the EXACT canonical patterns from the Roadmap Epigenomics portal.
# Verified against https://egg2.wustl.edu/roadmap/data/

# Histone marks and DNase: p-value signal tracks from MACS2
# Files are named: {EID}-{MARK}.pval.signal.bigwig
# All uppercase mark names, DNase uses "DNase" not "DNASE"
BASE_HISTONE = (
    "https://egg2.wustl.edu/roadmap/data/byFileType/"
    "signal/consolidated/macs2signal/pval/"
)

BASE_WGBS = (
    "https://egg2.wustl.edu/roadmap/data/byDataType/"
    "dnamethylation/WGBS/FractionalMethylation_bigwig/"
)


In [ ]:
# ── HTTP settings ─────────────────────────────────────────────────────────────
PROBE_TIMEOUT   = 10    # seconds per HEAD request
DOWNLOAD_CHUNK  = 1024 * 1024 * 4   # 4 MB chunks — good for bigWig files
MAX_RETRIES     = 3
RETRY_DELAY     = 5     # seconds between retries



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# URL construction
# ─────────────────────────────────────────────────────────────────────────────

def build_url(eid: str, mark: str) -> str:
    """
    Build the Roadmap Epigenomics download URL for one (epigenome, mark) pair.

    The URL patterns are:
      Histone/DNase: BASE_HISTONE + {EID}-{MARK}.pval.signal.bigwig
      WGBS:          BASE_WGBS    + {EID}_WGBS_FractionalMethylation.bigwig

    Note that 'DNase' follows the histone pattern even though it measures
    accessibility, not a histone modification. This is because Roadmap
    processed it with the same MACS2 pipeline as histone ChIP-seq.
    """
    if mark == WGBS_MARK:
        filename = f"{eid}_WGBS_FractionalMethylation.bigwig"
        return BASE_WGBS + filename
    else:
        filename = f"{eid}-{mark}.pval.signal.bigwig"
        return BASE_HISTONE + filename


def build_local_path(eid: str, mark: str) -> Path:
    """
    Construct the local save path for a bigWig file.
    Uses the same filename pattern as the remote URL.
    """
    if mark == WGBS_MARK:
        filename = f"{eid}_WGBS_FractionalMethylation.bigwig"
    else:
        filename = f"{eid}-{mark}.pval.signal.bigwig"
    return BIGWIG_DIR / filename


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# Phase 0: Probe URLs — build availability matrix without downloading anything
# ─────────────────────────────────────────────────────────────────────────────

def probe_all_urls() -> pd.DataFrame:
    """
    Send an HTTP HEAD request to every (epigenome × mark) URL.

    HEAD requests retrieve only the response headers — not the file body —
    so probing ~90 URLs costs almost no bandwidth and finishes in under 2 min.

    Returns a DataFrame with:
      - Index: epigenome IDs (e.g. 'E003')
      - Columns: mark names (e.g. 'H3K4me3', ..., 'WGBS')
      - Values: 1 (file exists, HTTP 200) or 0 (HTTP 404 or error)
    """
    eids  = list(EPIGENOMES.keys())
    avail = pd.DataFrame(0, index=eids, columns=ALL_MARKS, dtype=np.uint8)

    total = len(eids) * len(ALL_MARKS)
    print(f"\nProbing {total} URLs ({len(eids)} epigenomes × {len(ALL_MARKS)} marks)…")
    print("This takes ~60–120 seconds. No data is downloaded.\n")

    with tqdm(total=total, desc="Probing") as pbar:
        for eid in eids:
            for mark in ALL_MARKS:
                url = build_url(eid, mark)
                try:
                    # HEAD request: server returns headers only, no body.
                    # allow_redirects=True handles any HTTP → HTTPS redirects.
                    resp = requests.head(url, timeout=PROBE_TIMEOUT,
                                        allow_redirects=True)
                    if resp.status_code == 200:
                        avail.loc[eid, mark] = 1
                    # 404 = file genuinely doesn't exist for this epigenome
                    # Other codes (403, 500) treated as unavailable
                except requests.RequestException:
                    # Network timeout, DNS failure, etc. — treat as unavailable
                    pass
                pbar.update(1)

    return avail


def print_availability_report(avail: pd.DataFrame):
    """
    Print a human-readable summary of the availability matrix.
    Highlights which epigenomes are missing which marks.
    """
    print("\n" + "═" * 72)
    print("MARK AVAILABILITY MATRIX")
    print("═" * 72)

    # Print header
    header = f"{'EID':<8} {'Tissue':<22}"
    for mark in ALL_MARKS:
        header += f" {mark[:7]:<8}"
    header += "  Total"
    print(header)
    print("─" * 72)

    for eid, label in EPIGENOMES.items():
        row = avail.loc[eid]
        line = f"{eid:<8} {label[:22]:<22}"
        for mark in ALL_MARKS:
            symbol = "✅" if row[mark] == 1 else "❌"
            line += f" {symbol:<8}"
        line += f"  {int(row.sum())}/{len(ALL_MARKS)}"
        print(line)

    print("─" * 72)
    # Column totals
    totals_line = f"{'Total':<8} {'':<22}"
    for mark in ALL_MARKS:
        col_total = int(avail[mark].sum())
        totals_line += f" {col_total}/{len(EPIGENOMES):<6}"
    print(totals_line)
    print("═" * 72)

## Phase 1: Download available files


In [ ]:

def download_file(url: str, dest: Path, file_size_hint: int = 0) -> bool:
    """
    Download a single bigWig file with:
      - Resume support: if the file partially exists, we skip re-downloading
        completed bytes (HTTP Range request). This is critical for Google Drive
        which occasionally disconnects mid-download.
      - Progress bar per file
      - Automatic retry on network errors (up to MAX_RETRIES times)
      - Returns True if download succeeded, False otherwise

    Why streaming + chunks?
    bigWig files for Roadmap marks are typically 50–300 MB each.
    Loading the full response into memory before writing would crash
    a Colab session with 12 GB RAM when downloading 80+ files.
    Streaming writes directly to disk in 4 MB chunks.
    """
    for attempt in range(MAX_RETRIES):
        try:
            # ── Resume logic ──────────────────────────────────────────────
            # Check if a partial file already exists
            existing_bytes = dest.stat().st_size if dest.exists() else 0

            if existing_bytes > 0:
                # Ask the server to send only bytes we don't have yet
                headers = {'Range': f'bytes={existing_bytes}-'}
                resp = requests.get(url, headers=headers, stream=True,
                                    timeout=60)
                if resp.status_code == 416:
                    # 416 = Range Not Satisfiable = file already complete
                    return True
                if resp.status_code not in (200, 206):
                    # 206 = Partial Content (resume accepted by server)
                    # 200 = server doesn't support Range, starts from 0
                    existing_bytes = 0
                mode = 'ab'  # append to existing partial file
            else:
                resp = requests.get(url, stream=True, timeout=60)
                resp.raise_for_status()
                mode = 'wb'  # write fresh file

            # ── Total size for progress bar ───────────────────────────────
            total_size = (
                existing_bytes
                + int(resp.headers.get('Content-Length', 0))
            )

            # ── Stream write ──────────────────────────────────────────────
            with open(dest, mode) as f, tqdm(
                total=total_size,
                initial=existing_bytes,
                unit='B',
                unit_scale=True,
                unit_divisor=1024,
                desc=dest.name[:50],
                leave=False,
            ) as bar:
                for chunk in resp.iter_content(chunk_size=DOWNLOAD_CHUNK):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))

            return True

        except requests.RequestException as e:
            if attempt < MAX_RETRIES - 1:
                print(f"\n  Retry {attempt + 1}/{MAX_RETRIES} for {dest.name}: {e}")
                time.sleep(RETRY_DELAY)
            else:
                print(f"\n  FAILED after {MAX_RETRIES} attempts: {dest.name}")
                # Remove partial file so next run starts fresh
                if dest.exists():
                    dest.unlink()
                return False

    return False

In [ ]:
def download_all(avail: pd.DataFrame) -> dict:
    """
    Download all available files.

    Skips files that already exist at their full expected size.
    This makes the function safe to re-run after interruptions.

    Returns a dict {(eid, mark): local_path} for all successfully
    downloaded files.
    """
    # Build the list of (eid, mark) pairs that actually exist on the server
    to_download = [
        (eid, mark)
        for eid  in EPIGENOMES.keys()
        for mark in ALL_MARKS
        if avail.loc[eid, mark] == 1
    ]

    print(f"\nFiles to download: {len(to_download)}")
    print(f"Destination: {BIGWIG_DIR}\n")

    # Quick check: how many are already complete?
    already_done = sum(
        1 for eid, mark in to_download
        if build_local_path(eid, mark).exists()
        and build_local_path(eid, mark).stat().st_size > 10_000_000  # >10 MB
    )
    print(f"Already on disk: {already_done}/{len(to_download)}")

    results = {}
    failed  = []

    for i, (eid, mark) in enumerate(to_download):
        url  = build_url(eid, mark)
        dest = build_local_path(eid, mark)

        # ── Skip if file already looks complete ───────────────────────────
        # We use a 10 MB threshold as a rough "probably complete" check.
        # A proper check would compare against Content-Length from HEAD,
        # but that adds one HTTP request per file. The 10 MB heuristic is
        # fine because even the smallest valid bigWig files here are >50 MB.
        if dest.exists() and dest.stat().st_size > 10_000_000:
            results[(eid, mark)] = dest
            continue

        print(f"\n[{i+1}/{len(to_download)}] {eid} — {mark}")
        print(f"  URL: {url}")

        success = download_file(url, dest)

        if success:
            size_mb = dest.stat().st_size / 1e6
            print(f"  ✅ Saved: {dest.name} ({size_mb:.1f} MB)")
            results[(eid, mark)] = dest
        else:
            failed.append((eid, mark))

    # ── Final report ──────────────────────────────────────────────────────
    print("\n" + "═" * 60)
    print(f"DOWNLOAD COMPLETE")
    print(f"  Succeeded : {len(results)}")
    print(f"  Failed    : {len(failed)}")
    if failed:
        print("  Failed files:")
        for eid, mark in failed:
            print(f"    {eid} — {mark}")
    print("═" * 60)

    return results


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Main entry point
# ─────────────────────────────────────────────────────────────────────────────

def run_download_phase():
    """
    Execute Phase 1 completely:
      1. Probe all URLs → availability matrix
      2. Save availability matrix
      3. Download all available files
      4. Report final state
    """
    print("=" * 60)
    print("PHASE 1 — EPIGENETIC DATA DOWNLOAD")
    print("=" * 60)
    print(f"Epigenomes : {len(EPIGENOMES)}")
    print(f"Marks      : {len(ALL_MARKS)}")
    print(f"Max files  : {len(EPIGENOMES) * len(ALL_MARKS)}")

    # ── Step 1: Probe ─────────────────────────────────────────────────────
    avail = probe_all_urls()

    # ── Step 2: Save availability matrix ─────────────────────────────────
    avail.to_csv(AVAIL_PATH)
    print(f"\nAvailability matrix saved: {AVAIL_PATH}")

    # ── Step 3: Print report ──────────────────────────────────────────────
    print_availability_report(avail)

    # ── Step 4: Download ──────────────────────────────────────────────────
    # Prompt user before starting (Colab cell — comment out if running headless)
    total_available = int(avail.values.sum())
    print(f"\n{total_available} files available for download.")
    print("Starting download now…\n")

    download_results = download_all(avail)

    # ── Step 5: Build and return the mark_mask dict for Phase 2 ──────────
    # mark_mask[eid] is a uint8 array of shape (9,)
    # 1 = mark available and downloaded, 0 = not available
    mark_mask = {}
    for eid in EPIGENOMES.keys():
        mask = np.array(
            [1 if (eid, mark) in download_results else 0
             for mark in ALL_MARKS],
            dtype=np.uint8,
        )
        mark_mask[eid] = mask

    # Save mark_mask as a numpy file for Phase 2
    mask_save_path = PROC_DIR / 'mark_mask.npy'
    np.save(mask_save_path, mark_mask)
    print(f"\nMark masks saved: {mask_save_path}")

    return avail, download_results, mark_mask


if __name__ == '__main__':
    avail, results, mark_mask = run_download_phase()

PHASE 1 — EPIGENETIC DATA DOWNLOAD
Epigenomes : 10
Marks      : 9
Max files  : 90

Probing 90 URLs (10 epigenomes × 9 marks)…
This takes ~60–120 seconds. No data is downloaded.



Probing: 100%|██████████| 90/90 [00:12<00:00,  7.02it/s]



Availability matrix saved: /content/drive/MyDrive/PFE/data/epigenetics/mark_availability.csv

════════════════════════════════════════════════════════════════════════
MARK AVAILABILITY MATRIX
════════════════════════════════════════════════════════════════════════
EID      Tissue                 H3K4me3  H3K4me1  H3K36me  H3K27me  H3K9me3  H3K27ac  H3K9ac   DNase    WGBS      Total
────────────────────────────────────────────────────────────────────────
E003     H1_ESC                 ✅        ✅        ✅        ✅        ✅        ✅        ✅        ✅        ✅         9/9
E017     IMR90_fetal_lung       ✅        ✅        ✅        ✅        ✅        ✅        ✅        ✅        ✅         9/9
E050     brain_angular_gyrus    ✅        ✅        ✅        ✅        ✅        ✅        ❌        ✅        ✅         8/9
E067     brain_cortex           ✅        ✅        ✅        ✅        ✅        ✅        ✅        ❌        ❌         7/9
E062     monocytes              ✅        ✅        ✅        ✅        ✅

  ✅ Saved: E003-H3K4me3.pval.signal.bigwig (628.4 MB)

[2/75] E003 — H3K4me1
  URL: https://egg2.wustl.edu/roadmap/data/byFileType/signal/consolidated/macs2signal/pval/E003-H3K4me1.pval.signal.bigwig


  ✅ Saved: E003-H3K4me1.pval.signal.bigwig (707.7 MB)

[3/75] E003 — H3K36me3
  URL: https://egg2.wustl.edu/roadmap/data/byFileType/signal/consolidated/macs2signal/pval/E003-H3K36me3.pval.signal.bigwig


...

  ✅ Saved: E116-H3K27me3.pval.signal.bigwig (349.0 MB)

[72/75] E116 — H3K9me3
  URL: https://egg2.wustl.edu/roadmap/data/byFileType/signal/consolidated/macs2signal/pval/E116-H3K9me3.pval.signal.bigwig


  ✅ Saved: E116-H3K9me3.pval.signal.bigwig (346.8 MB)

[73/75] E116 — H3K27ac
  URL: https://egg2.wustl.edu/roadmap/data/byFileType/signal/consolidated/macs2signal/pval/E116-H3K27ac.pval.signal.bigwig


  ✅ Saved: E116-H3K27ac.pval.signal.bigwig (343.8 MB)

[74/75] E116 — H3K9ac
  URL: https://egg2.wustl.edu/roadmap/data/byFileType/signal/consolidated/macs2signal/pval/E116-H3K9ac.pval.signal.bigwig


  ✅ Saved: E116-H3K9ac.pval.signal.bigwig (350.2 MB)

[75/75] E116 — DNase
  URL: https://egg2.wustl.edu/roadmap/data/byFileType/signal/consolidated/macs2signal/pval/E116-DNase.pval.signal.bigwig


  ✅ Saved: E116-DNase.pval.signal.bigwig (799.0 MB)

════════════════════════════════════════════════════════════
DOWNLOAD COMPLETE
  Succeeded : 75
  Failed    : 0
════════════════════════════════════════════════════════════

Mark masks saved: /content/drive/MyDrive/PFE/data/epigenetics/processed/mark_mask.npy


In [ ]:
from pathlib import Path

BIGWIG_DIR = Path('/content/drive/MyDrive/PFE/data/epigenetics/bigwig')

files = list(BIGWIG_DIR.glob("*"))

print("Number of files:", len(files))

for f in files[:5]:
    print(f.name, f.stat().st_size / 1e6, "MB")

Number of files: 75
E062-H3K4me1.pval.signal.bigwig 704.438001 MB
E003_WGBS_FractionalMethylation.bigwig 336.112394 MB
E104_WGBS_FractionalMethylation.bigwig 348.643492 MB
E067-H3K4me3.pval.signal.bigwig 686.907057 MB
E067-H3K4me1.pval.signal.bigwig 692.087157 MB


In [ ]:
!ls -lh /content/drive/MyDrive/PFE/data/epigenetics/bigwig | head

total 42G
-rw-r--r-- 1 root root 1.2G May 12 14:18 E003-DNase.pval.signal.bigwig
-rw-r--r-- 1 root root 589M May 12 14:11 E003-H3K27ac.pval.signal.bigwig
-rw-r--r-- 1 root root 644M May 12 14:06 E003-H3K27me3.pval.signal.bigwig
-rw-r--r-- 1 root root 652M May 12 14:03 E003-H3K36me3.pval.signal.bigwig
-rw-r--r-- 1 root root 675M May 12 14:01 E003-H3K4me1.pval.signal.bigwig
-rw-r--r-- 1 root root 600M May 12 13:58 E003-H3K4me3.pval.signal.bigwig
-rw-r--r-- 1 root root 670M May 12 14:13 E003-H3K9ac.pval.signal.bigwig
-rw-r--r-- 1 root root 644M May 12 14:08 E003-H3K9me3.pval.signal.bigwig
-rw-r--r-- 1 root root 321M May 12 14:20 E003_WGBS_FractionalMethylation.bigwig


In [ ]:
import os

path = "/content/drive/MyDrive/PFE/data/epigenetics/bigwig"
print("Exists:", os.path.exists(path))
print("Files:", len(os.listdir(path)))

Exists: True
Files: 75


In [ ]:
!rm -rf /content/local_epigenetics_backup

In [ ]:
!ls "/content/drive/MyDrive/PFE/data/epigenetics/bigwig" | head

E003-DNase.pval.signal.bigwig
E003-H3K27ac.pval.signal.bigwig
E003-H3K27me3.pval.signal.bigwig
E003-H3K36me3.pval.signal.bigwig
E003-H3K4me1.pval.signal.bigwig
E003-H3K4me3.pval.signal.bigwig
E003-H3K9ac.pval.signal.bigwig
E003-H3K9me3.pval.signal.bigwig
E003_WGBS_FractionalMethylation.bigwig
E017-DNase.pval.signal.bigwig


In [ ]:
import os

path = "/content/drive/MyDrive/PFE/data/"

files = os.listdir(path)
print("Number of files:", len(files))
print(files[:10])

Number of files: 1
['epigenetics']


In [ ]:
import shutil
import os
from pathlib import Path
from tqdm import tqdm

# Where the files actually are (Colab local disk)
# Check both common locations
for candidate in ['/content/bigwig', '/content/data/epigenetics/bigwig',
                  '/tmp/bigwig']:
    p = Path(candidate)
    if p.exists():
        files = list(p.glob("*.bigwig"))
        print(f"Found {len(files)} bigwig files at: {candidate}")
        break
else:
    # They might already be in Drive — let's check all bigwig files
    result = __import__('subprocess').run(
        ['find', '/content', '-name', '*.bigwig', '-not', '-path', '*/drive/*'],
        capture_output=True, text=True
    )
    print("Bigwig files on local disk:")
    print(result.stdout[:3000] if result.stdout else "NONE FOUND ON LOCAL DISK")

    # Also check what the download script actually used
    print("\nChecking what BIGWIG_DIR was set to in the download script...")

Bigwig files on local disk:
NONE FOUND ON LOCAL DISK

Checking what BIGWIG_DIR was set to in the download script...


In [ ]:
# Once you know the source location, copy to Drive immediately
# Replace SOURCE_DIR with whatever the search found above

SOURCE_DIR = Path('/content/drive/MyDrive/PFE/data/epigenetics/bigwig')
# ^ If files are already here (from your glob check), nothing to copy.
# The issue might just be that the previous script already wrote to Drive correctly.

# Verify by checking total size
total_size_gb = sum(f.stat().st_size for f in SOURCE_DIR.glob("*.bigwig")) / 1e9
print(f"Total bigwig size on disk: {total_size_gb:.1f} GB")
print(f"Number of files: {len(list(SOURCE_DIR.glob('*.bigwig')))}")

Total bigwig size on disk: 44.6 GB
Number of files: 75


In [ ]:
# Your sequence-phase outputs are on Drive but not where the script expects.
# The find command showed nothing — but that searched only /content/drive/MyDrive
# Let me search more carefully including different PFE folder locations

import subprocess
from pathlib import Path

# Search entire Drive, not just PFE subfolder
searches = [
    'window_index.parquet',
    'window_embeddings.npy',
    'nt_checkpoint.h5',
    'clean_windows.parquet',
    'island_masks.npy',
]

for filename in searches:
    result = subprocess.run(
        ['find', '/content/drive', '-name', filename, '-type', 'f'],
        capture_output=True, text=True, timeout=120
    )
    if result.stdout.strip():
        print(f"FOUND {filename}:")
        print(f"  {result.stdout.strip()}")
    else:
        print(f"NOT FOUND: {filename}")

NOT FOUND: window_index.parquet
NOT FOUND: window_embeddings.npy
NOT FOUND: nt_checkpoint.h5
NOT FOUND: clean_windows.parquet
NOT FOUND: island_masks.npy


In [ ]:
# Also list every folder that exists under your Drive root
# to find where the sequence work actually landed

print("All folders under MyDrive (2 levels deep):")
drive = Path('/content/drive/MyDrive')
for item in sorted(drive.rglob('*')):
    if item.is_dir() and len(item.relative_to(drive).parts) <= 2:
        n_files = len(list(item.glob('*')))
        print(f"  {item.relative_to(drive)}/  ({n_files} items)")

All folders under MyDrive (2 levels deep):
  PFE/  (1 items)
  PFE/data/  (1 items)


In [ ]:
# Write a tiny test file, wait 30 seconds, check Drive UI, then delete it.
# If it appears in the UI, your bigwig files are there too — just not indexed yet.

test_path = Path('/content/drive/MyDrive/PFE/data/epigenetics/SYNC_TEST.txt')
test_path.write_text("If you see this in Drive UI, all bigwig files are safe.")
print(f"Test file written: {test_path}")
print(f"Size: {test_path.stat().st_size} bytes")
print("\nWait 1-2 minutes then check Google Drive web UI.")
print("If SYNC_TEST.txt appears, your 44.6 GB of bigwig files are confirmed safe.")

Test file written: /content/drive/MyDrive/PFE/data/epigenetics/SYNC_TEST.txt
Size: 55 bytes

Wait 1-2 minutes then check Google Drive web UI.
If SYNC_TEST.txt appears, your 44.6 GB of bigwig files are confirmed safe.


In [ ]:
from pathlib import Path
import os

print("=" * 60)
print("MOUNT DIAGNOSTIC")
print("=" * 60)

# Check 1: Is Drive actually mounted?
drive_mount = Path('/content/drive')
mydrive = Path('/content/drive/MyDrive')

print(f"\n/content/drive exists   : {drive_mount.exists()}")
print(f"/content/drive/MyDrive  : {mydrive.exists()}")

# Check 2: What is at /content/drive/MyDrive?
if mydrive.exists():
    items = list(mydrive.iterdir())
    print(f"Contents of MyDrive     : {[i.name for i in items]}")
else:
    print("MyDrive NOT ACCESSIBLE")

# Check 3: Where did the test file actually go?
# Search for it on both local disk and drive
test_candidates = [
    Path('/content/drive/MyDrive/PFE/data/epigenetics/SYNC_TEST.txt'),
    Path('/content/SYNC_TEST.txt'),
    Path('/content/PFE/data/epigenetics/SYNC_TEST.txt'),
]
print("\nSearching for SYNC_TEST.txt:")
for p in test_candidates:
    print(f"  {p}: {'EXISTS' if p.exists() else 'not found'}")

# Check 4: Where are the bigwig files ACTUALLY stored?
import subprocess
result = subprocess.run(
    ['find', '/content', '-name', '*.bigwig', '-type', 'f'],
    capture_output=True, text=True, timeout=60
)
lines = result.stdout.strip().split('\n') if result.stdout.strip() else []
print(f"\nBigwig files found on ALL of /content: {len(lines)}")
if lines and lines[0]:
    # Show first 3 paths to understand the structure
    for line in lines[:3]:
        print(f"  {line}")
    print(f"  ... and {len(lines)-3} more" if len(lines) > 3 else "")

    # Critical check: are these on Drive or local?
    on_drive = [l for l in lines if '/content/drive/' in l]
    on_local = [l for l in lines if '/content/drive/' not in l]
    print(f"\n  ON DRIVE (safe) : {len(on_drive)}")
    print(f"  ON LOCAL (at risk): {len(on_local)}")
    if on_local:
        print(f"\n  ⚠️  LOCAL PATHS (will be lost on restart):")
        for l in on_local[:3]:
            print(f"    {l}")

# Check 5: The BIGWIG_DIR that the download script used
print("\nThe BIGWIG_DIR constant in download script was:")
print("  Path('/content/drive/MyDrive/PFE/data/epigenetics/bigwig')")
bw_dir = Path('/content/drive/MyDrive/PFE/data/epigenetics/bigwig')
print(f"  Exists now: {bw_dir.exists()}")
if bw_dir.exists():
    n = len(list(bw_dir.glob('*.bigwig')))
    print(f"  Files in it: {n}")

MOUNT DIAGNOSTIC

/content/drive exists   : True
/content/drive/MyDrive  : True
Contents of MyDrive     : ['PFE']

Searching for SYNC_TEST.txt:
  /content/drive/MyDrive/PFE/data/epigenetics/SYNC_TEST.txt: EXISTS
  /content/SYNC_TEST.txt: not found
  /content/PFE/data/epigenetics/SYNC_TEST.txt: not found

Bigwig files found on ALL of /content: 75
  /content/drive/MyDrive/PFE/data/epigenetics/bigwig/E062-H3K4me1.pval.signal.bigwig
  /content/drive/MyDrive/PFE/data/epigenetics/bigwig/E003_WGBS_FractionalMethylation.bigwig
  /content/drive/MyDrive/PFE/data/epigenetics/bigwig/E104_WGBS_FractionalMethylation.bigwig
  ... and 72 more

  ON DRIVE (safe) : 75
  ON LOCAL (at risk): 0

The BIGWIG_DIR constant in download script was:
  Path('/content/drive/MyDrive/PFE/data/epigenetics/bigwig')
  Exists now: True
  Files in it: 75


In [ ]:
# Check 6: Test write to Drive and verify it actually lands there
print("\n" + "=" * 60)
print("WRITE TEST")
print("=" * 60)

# Try writing to Drive path explicitly
test_on_drive = Path('/content/drive/MyDrive/PFE/data/epigenetics/WRITE_TEST.txt')
test_on_local = Path('/content/WRITE_TEST.txt')

# Write to Drive path
test_on_drive.write_text("drive test")
test_on_local.write_text("local test")

# Check where each actually landed
print(f"\nWrote to Drive path: {test_on_drive}")
print(f"  File exists at that path: {test_on_drive.exists()}")
print(f"  Size: {test_on_drive.stat().st_size} bytes")

# Now check if the drive path resolves to the same inode as a local path
# This would indicate the drive mount is broken/redirected
import os
try:
    drive_stat = os.stat(str(test_on_drive))
    print(f"  inode: {drive_stat.st_ino}")
    print(f"  device: {drive_stat.st_dev}")
except Exception as e:
    print(f"  stat failed: {e}")

# Check what filesystem /content/drive is
result = subprocess.run(['df', '-T', '/content/drive/MyDrive'],
                       capture_output=True, text=True)
print(f"\nFilesystem info for /content/drive/MyDrive:")
print(result.stdout)

result2 = subprocess.run(['df', '-T', '/content'],
                        capture_output=True, text=True)
print(f"Filesystem info for /content:")
print(result2.stdout)


WRITE TEST

Wrote to Drive path: /content/drive/MyDrive/PFE/data/epigenetics/WRITE_TEST.txt
  File exists at that path: True
  Size: 10 bytes
  inode: 4976
  device: 54

Filesystem info for /content/drive/MyDrive:
Filesystem     Type    1K-blocks     Used Available Use% Mounted on
overlay        overlay 112947452 84967280  27963788  76% /

Filesystem info for /content:
Filesystem     Type    1K-blocks     Used Available Use% Mounted on
overlay        overlay 112947452 84967280  27963788  76% /



In [ ]:
# The drive.mount() call will fail if /content/drive already exists
# with content in it. We need to handle this carefully.

import os
import shutil
from pathlib import Path
from google.colab import drive

# Check if the fake drive directory exists
fake_drive = Path('/content/drive')
print(f"Fake /content/drive exists: {fake_drive.exists()}")
print(f"Contents: {list(fake_drive.iterdir()) if fake_drive.exists() else 'empty'}")

Fake /content/drive exists: True
Contents: [PosixPath('/content/drive/MyDrive')]


In [ ]:
# CRITICAL: Before moving anything, verify the bigwig files are still there
bw_dir = Path('/content/drive/MyDrive/PFE/data/epigenetics/bigwig')
bw_files = list(bw_dir.glob('*.bigwig'))
total_gb = sum(f.stat().st_size for f in bw_files) / 1e9
print(f"Bigwig files confirmed present: {len(bw_files)} files, {total_gb:.1f} GB")
print("DO NOT RESTART RUNTIME UNTIL THESE ARE ON REAL DRIVE")

Bigwig files confirmed present: 75 files, 44.6 GB
DO NOT RESTART RUNTIME UNTIL THESE ARE ON REAL DRIVE


In [ ]:
# Step 1a: Move the bigwig files to a safe temporary location
# that won't be affected when we fix the drive mount

import shutil

# Move to /tmp which is guaranteed local and won't be touched by mount operations
TEMP_BIGWIG = Path('/tmp/bigwig_backup')
TEMP_BIGWIG.mkdir(exist_ok=True)

print(f"Moving {len(bw_files)} bigwig files to /tmp/bigwig_backup...")
print("This is a local move (fast, no network) — just changing directory location")

for f in bw_files:
    dest = TEMP_BIGWIG / f.name
    if not dest.exists():
        shutil.move(str(f), str(dest))

moved = list(TEMP_BIGWIG.glob('*.bigwig'))
print(f"Moved: {len(moved)} files")
print(f"Total size: {sum(f.stat().st_size for f in moved)/1e9:.1f} GB")
print("\nFiles are safe in /tmp/bigwig_backup")
print("Proceeding to fix Drive mount...")

Moving 75 bigwig files to /tmp/bigwig_backup...
This is a local move (fast, no network) — just changing directory location
Moved: 75 files
Total size: 44.6 GB

Files are safe in /tmp/bigwig_backup
Proceeding to fix Drive mount...


In [ ]:
# Step 1b: Remove the fake /content/drive directory so real mount can proceed
# We already moved the bigwig files to /tmp so this is safe

import subprocess

# Remove the fake drive directory
result = subprocess.run(['rm', '-rf', '/content/drive'],
                       capture_output=True, text=True)
print(f"Removed fake /content/drive: {result.returncode == 0}")
print(f"Exists now: {Path('/content/drive').exists()}")

Removed fake /content/drive: True
Exists now: False


In [ ]:
# Step 1c: Mount the real Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify it's now a real FUSE mount
result = subprocess.run(['df', '-T', '/content/drive/MyDrive'],
                       capture_output=True, text=True)
print("Filesystem after mount:")
print(result.stdout)
# Should now show: fuse   (not overlay)
# And a different size than /content (15 GB vs Drive's quota)

Mounted at /content/drive
Filesystem after mount:
Filesystem     Type       1K-blocks     Used Available Use% Mounted on
drive          fuse.drive 112947452 86384348  26563104  77% /content/drive



In [ ]:
# Now copy from /tmp to the real Drive
import shutil
from pathlib import Path
from tqdm import tqdm

TEMP_BIGWIG   = Path('/tmp/bigwig_backup')
REAL_BIGWIG   = Path('/content/drive/MyDrive/PFE/data/epigenetics/bigwig')
REAL_BIGWIG.mkdir(parents=True, exist_ok=True)

temp_files = list(TEMP_BIGWIG.glob('*.bigwig'))
print(f"Files to copy to Drive: {len(temp_files)}")
print(f"Total size: {sum(f.stat().st_size for f in temp_files)/1e9:.1f} GB")
print("\nCopying to Drive — this will take 30-90 minutes depending on Drive upload speed")
print("DO NOT close this tab or let the session time out\n")

failed = []
for i, src in enumerate(tqdm(temp_files, desc="Uploading to Drive")):
    dest = REAL_BIGWIG / src.name

    # Skip if already there and correct size
    if dest.exists() and dest.stat().st_size == src.stat().st_size:
        continue

    try:
        shutil.copy2(str(src), str(dest))

        # Verify the copy landed correctly
        if dest.stat().st_size != src.stat().st_size:
            print(f"\n  ⚠️  Size mismatch: {src.name}")
            failed.append(src.name)
    except Exception as e:
        print(f"\n  ❌ Failed: {src.name}: {e}")
        failed.append(src.name)

# Final verification
drive_files = list(REAL_BIGWIG.glob('*.bigwig'))
total_drive_gb = sum(f.stat().st_size for f in drive_files) / 1e9
print(f"\n{'='*50}")
print(f"Files now on real Drive: {len(drive_files)}")
print(f"Total on Drive: {total_drive_gb:.1f} GB")
print(f"Failed: {len(failed)}")
if failed:
    print("Failed files:", failed)
else:
    print("✅ All bigwig files safely on Google Drive")

Files to copy to Drive: 75
Total size: 44.6 GB

Copying to Drive — this will take 30-90 minutes depending on Drive upload speed
DO NOT close this tab or let the session time out



Uploading to Drive: 100%|██████████| 75/75 [10:09<00:00,  8.13s/it]


Files now on real Drive: 75
Total on Drive: 44.6 GB
Failed: 0
✅ All bigwig files safely on Google Drive


In [ ]:
# Confirm sequence-phase files are now visible from this notebook
from pathlib import Path

PROC_DIR = Path('/content/drive/MyDrive/PFE/data/processed')
expected = [
    'window_index.parquet',
    'window_embeddings.npy',
    'clean_windows.parquet',
    'island_masks.npy',
    'nt_checkpoint.h5',
]

print("Sequence-phase files on real Drive:")
for name in expected:
    p = PROC_DIR / name
    if p.exists():
        print(f"  ✅ {name:<30} {p.stat().st_size/1e6:.1f} MB")
    else:
        print(f"  ❌ {name:<30} NOT FOUND")

# Verify mark_availability.csv is also accessible
avail = Path('/content/drive/MyDrive/PFE/data/epigenetics/mark_availability.csv')
print(f"\n  {'✅' if avail.exists() else '❌'} mark_availability.csv")

Sequence-phase files on real Drive:
  ✅ window_index.parquet           2.1 MB
  ✅ window_embeddings.npy          61.0 MB
  ✅ clean_windows.parquet          29.1 MB
  ✅ island_masks.npy               61.0 MB
  ✅ nt_checkpoint.h5               22.2 MB

  ❌ mark_availability.csv


In [ ]:
from pathlib import Path

# ── Verify mount is real (corrected check) ────────────────────────────────────
import subprocess

result = subprocess.run(
    ['df', '-T', '/content/drive/MyDrive'],
    capture_output=True, text=True
)
fs_line = result.stdout.strip().split('\n')[-1]
fs_type = fs_line.split()[1]   # second column is Type

print(f"Filesystem type: {fs_type}")
assert 'fuse' in fs_type.lower(), f"Drive not mounted! Got: {fs_type}"
print("✅ Drive is real (fuse.drive confirmed)\n")

# ── Check all critical files ──────────────────────────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/PFE')
PROC_DIR   = DRIVE_ROOT / 'data' / 'processed'
EPI_DIR    = DRIVE_ROOT / 'data' / 'epigenetics'
BIGWIG_DIR = EPI_DIR / 'bigwig'

files_to_check = {
    # Sequence phase
    'window_index.parquet' : PROC_DIR / 'window_index.parquet',
    'window_embeddings.npy': PROC_DIR / 'window_embeddings.npy',
    'clean_windows.parquet': PROC_DIR / 'clean_windows.parquet',
    'island_masks.npy'     : PROC_DIR / 'island_masks.npy',
    'nt_checkpoint.h5'     : PROC_DIR / 'nt_checkpoint.h5',
    # Epigenetics phase
    'mark_availability.csv': EPI_DIR / 'mark_availability.csv',
    'mark_mask.npy'        : EPI_DIR / 'processed' / 'mark_mask.npy',
    'bigwig_dir (75 files)': BIGWIG_DIR,
}

all_ok = True
for name, path in files_to_check.items():
    if path.is_dir():
        n     = len(list(path.glob('*.bigwig')))
        size  = sum(f.stat().st_size for f in path.glob('*.bigwig')) / 1e9
        ok    = n == 75
        print(f"  {'✅' if ok else '❌'} {name:<28} {n} files  {size:.1f} GB")
    elif path.exists():
        size_mb = path.stat().st_size / 1e6
        print(f"  ✅ {name:<28} {size_mb:.1f} MB")
    else:
        print(f"  ❌ {name:<28} MISSING — {path}")
        all_ok = False

print()
if all_ok:
    print("✅ All files present. Ready to run alignment script.")
else:
    print("Some files missing — see below.")

Filesystem type: fuse.drive
✅ Drive is real (fuse.drive confirmed)

  ✅ window_index.parquet         2.1 MB
  ✅ window_embeddings.npy        61.0 MB
  ✅ clean_windows.parquet        29.1 MB
  ✅ island_masks.npy             61.0 MB
  ✅ nt_checkpoint.h5             22.2 MB
  ✅ mark_availability.csv        0.0 MB
  ✅ mark_mask.npy                0.0 MB
  ✅ bigwig_dir (75 files)        75 files  44.6 GB

✅ All files present. Ready to run alignment script.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/PFE')
EPI_DIR    = DRIVE_ROOT / 'data' / 'epigenetics'
PROC_DIR   = EPI_DIR / 'processed'
PROC_DIR.mkdir(parents=True, exist_ok=True)

ALL_MARKS = [
    'H3K4me3', 'H3K4me1', 'H3K36me3', 'H3K27me3', 'H3K9me3',
    'H3K27ac', 'H3K9ac', 'DNase', 'WGBS',
]

# Load the availability matrix you just saved
avail = pd.read_csv(EPI_DIR / 'mark_availability.csv', index_col=0)

# Build mark_mask: shape (N_epigenomes, N_marks), dtype uint8
# Row order matches EPIGENOMES dict order, column order matches ALL_MARKS
mark_mask_matrix = avail[ALL_MARKS].values.astype(np.uint8)

# Save
mask_path = PROC_DIR / 'mark_mask_matrix.npy'
np.save(mask_path, mark_mask_matrix)
print(f"Saved mark_mask_matrix.npy: {mask_path}")
print(f"Shape: {mark_mask_matrix.shape}  (expected (10, 9))")
print(f"Size:  {mask_path.stat().st_size} bytes")

# Also save the per-epigenome dict version that the download script produced
# as mark_mask.npy — the alignment script references both names
eids      = list(avail.index)
mark_mask = {eid: avail.loc[eid, ALL_MARKS].values.astype(np.uint8)
             for eid in eids}

mask_dict_path = PROC_DIR / 'mark_mask.npy'
np.save(mask_dict_path, mark_mask)
print(f"Saved mark_mask.npy (dict):  {mask_dict_path}")
print(f"Size: {mask_dict_path.stat().st_size} bytes")

# Quick sanity check — print each epigenome's mask
print("\nPer-epigenome mark availability:")
print(f"{'EID':<8}", end="")
for m in ALL_MARKS:
    print(f" {m[:7]:<8}", end="")
print()
for eid in eids:
    print(f"{eid:<8}", end="")
    for val in mark_mask[eid]:
        print(f" {'✅' if val else '❌':<8}", end="")
    print(f"  {mark_mask[eid].sum()}/9")

Saved mark_mask_matrix.npy: /content/drive/MyDrive/PFE/data/epigenetics/processed/mark_mask_matrix.npy
Shape: (10, 9)  (expected (10, 9))
Size:  218 bytes
Saved mark_mask.npy (dict):  /content/drive/MyDrive/PFE/data/epigenetics/processed/mark_mask.npy
Size: 771 bytes

Per-epigenome mark availability:
EID      H3K4me3  H3K4me1  H3K36me  H3K27me  H3K9me3  H3K27ac  H3K9ac   DNase    WGBS    
E003     ✅        ✅        ✅        ✅        ✅        ✅        ✅        ✅        ✅         9/9
E017     ✅        ✅        ✅        ✅        ✅        ✅        ✅        ✅        ✅         9/9
E050     ✅        ✅        ✅        ✅        ✅        ✅        ❌        ✅        ✅         8/9
E067     ✅        ✅        ✅        ✅        ✅        ✅        ✅        ❌        ❌         7/9
E062     ✅        ✅        ✅        ✅        ✅        ✅        ✅        ❌        ❌         7/9
E045     ✅        ✅        ✅        ✅        ✅        ✅        ❌        ❌        ❌         6/9
E065     ✅        ✅        ✅        ✅ 

In [ ]:
import pandas as pd
import numpy as np

# Reconstruct mark_availability.csv from what we know is on Drive.
# A file existing in bigwig/ IS the ground truth for availability.

EPIGENOMES = {
    'E003': 'H1_ESC',
    'E017': 'IMR90_fetal_lung',
    'E050': 'brain_angular_gyrus',
    'E067': 'brain_cortex',
    'E062': 'monocytes',
    'E045': 'HSC',
    'E065': 'aorta',
    'E071': 'brain_hippocampus',
    'E104': 'heart_right_atrium',
    'E116': 'GM12878_lymphoblastoid',
}

ALL_MARKS = [
    'H3K4me3', 'H3K4me1', 'H3K36me3', 'H3K27me3', 'H3K9me3',
    'H3K27ac', 'H3K9ac', 'DNase', 'WGBS',
]

def file_exists_on_drive(eid, mark):
    """Check whether the bigwig file for this (epigenome, mark) is on Drive."""
    if mark == 'WGBS':
        fname = f"{eid}_WGBS_FractionalMethylation.bigwig"
    else:
        fname = f"{eid}-{mark}.pval.signal.bigwig"
    path = BIGWIG_DIR / fname
    return 1 if (path.exists() and path.stat().st_size > 10_000_000) else 0

# Build the matrix by checking actual files on Drive
avail = pd.DataFrame(
    {mark: [file_exists_on_drive(eid, mark) for eid in EPIGENOMES]
     for mark in ALL_MARKS},
    index=list(EPIGENOMES.keys()),
    dtype=np.uint8,
)
avail.index.name = 'eid'

# Save to Drive
avail_path = EPI_DIR / 'mark_availability.csv'
avail.to_csv(avail_path)
print(f"Saved: {avail_path}  ({avail_path.stat().st_size} bytes)")

# Print the matrix so you can verify it matches what was shown earlier
print("\nMark availability matrix (1=file on Drive, 0=missing):")
print(avail.to_string())

total = int(avail.values.sum())
print(f"\nTotal available (epigenome, mark) pairs: {total}  (expected 75)")

Saved: /content/drive/MyDrive/PFE/data/epigenetics/mark_availability.csv  (302 bytes)

Mark availability matrix (1=file on Drive, 0=missing):
      H3K4me3  H3K4me1  H3K36me3  H3K27me3  H3K9me3  H3K27ac  H3K9ac  DNase  WGBS
eid                                                                              
E003        1        1         1         1        1        1       1      1     1
E017        1        1         1         1        1        1       1      1     1
E050        1        1         1         1        1        1       0      1     1
E067        1        1         1         1        1        1       1      0     0
E062        1        1         1         1        1        1       1      0     0
E045        1        1         1         1        1        1       0      0     0
E065        1        1         1         1        1        1       0      0     1
E071        1        1         1         1        1        1       0      0     1
E104        1        1         1      

# Phase 2 — Signal Extraction & Alignment

---




What this script does
---------------------
1. Loads window coordinates from window_index.parquet (your existing 28K tiles)
2. For every (epigenome × mark) bigWig file, extracts 128-bin signal per tile
3. Handles None bins, clips and normalizes signals correctly per mark type
4. Writes results into a structured HDF5 file
5. Constructs and saves both mark-level and bin-level masks
6. Runs biological sanity checks

Why 128 bins of 16 bp?
-----------------------
2048 bp / 128 = 16 bp per bin. This resolution is:
- Fine enough to distinguish TSS peaks (H3K4me3 ~200 bp) from broad marks
- Coarse enough that extraction is fast (~0.5 ms per tile per mark)
- Standard in the field (DeepSEA, Enformer use similar resolutions)

The CNN will see (9, 128) per tile — 9 channels (marks) × 128 positions.

Why HDF5 and not a .npy array?
--------------------------------
With 10 epigenomes × 28K tiles × 9 marks × 128 bins × float16:
  10 × 28000 × 9 × 128 × 2 bytes = ~650 MB

Storing this as a flat numpy array requires loading all 650 MB to read
one tile. HDF5 with chunking lets you read one epigenome's worth of data
(65 MB) or one mark's column without loading the rest.

The HDF5 structure produced:
------------------------------


```
 epi_signals.h5
├── attrs: n_tiles, n_bins=128, marks_order, epigenome_ids, window_size
├── E003/
│   ├── H3K4me3   (N_tiles, 128) float16   ← -log10(p-val), clipped [0,50]
│   ├── H3K4me1   (N_tiles, 128) float16
│   ├── H3K36me3  (N_tiles, 128) float16
│   ├── H3K27me3  (N_tiles, 128) float16
│   ├── H3K9me3   (N_tiles, 128) float16
│   ├── H3K27ac   (N_tiles, 128) float16
│   ├── H3K9ac    (N_tiles, 128) float16
│   ├── DNase     (N_tiles, 128) float16
│   └── WGBS      (N_tiles, 128) float16   ← fractional methylation [0,1]
├── E017/
│   └── ...
└── ...
```



Masks produced:
----------------
mark_mask_matrix.npy — shape (N_epigenomes, N_marks), uint8
  1 = mark available for this epigenome, 0 = missing

bin_mask.h5 — shape per (epigenome, mark) = (N_tiles, 128), uint8
  1 = valid signal in this bin, 0 = pyBigWig returned None (assembly gap)
  (In practice >99.9% of bins are 1 for canonical chromosomes)

In [ ]:
!pip install pyBigWig

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.0/187.0 kB 3.2 MB/s eta 0:00:00


In [ ]:
import os
import json
import shutil
import time
import numpy as np
import pandas as pd
import h5py
import pyBigWig
from pathlib import Path
from tqdm import tqdm


In [ ]:
# Configuration — set MODE before running
# ─────────────────────────────────────────────────────────────────────────────

MODE = 'COLAB'   # 'COLAB' or 'LOCAL'

# ── Colab paths ───────────────────────────────────────────────────────────────
DRIVE_ROOT   = Path('/content/drive/MyDrive/PFE')
DRIVE_BIGWIG = DRIVE_ROOT / 'data' / 'epigenetics' / 'bigwig'
DRIVE_PROC   = DRIVE_ROOT / 'data' / 'epigenetics' / 'processed'
DRIVE_SEQ    = DRIVE_ROOT / 'data' / 'processed'

# Local staging area (fast SSD on Colab, or your local machine's path)
# On Colab: /tmp has ~100 GB of fast local NVMe
# On local machine: change to wherever you want temp files
LOCAL_BIGWIG = Path('/tmp/bigwig_staged')
LOCAL_H5     = Path('/tmp/epi_signals_wip.h5')      # work-in-progress HDF5
LOCAL_MASK   = Path('/tmp/bin_mask_wip.h5')

# Final output destinations (always on Drive for Colab, or local for MODE B)
if MODE == 'COLAB':
    BIGWIG_SOURCE = DRIVE_BIGWIG    # where to copy FROM
    FINAL_H5      = DRIVE_PROC / 'epi_signals.h5'
    FINAL_MASK    = DRIVE_PROC / 'bin_mask.h5'
    WINDOW_PATH   = DRIVE_SEQ / 'window_index.parquet'
    AVAIL_PATH    = DRIVE_ROOT / 'data' / 'epigenetics' / 'mark_availability.csv'
    CHECKPOINT    = Path('/tmp/extraction_checkpoint.json')
else:
    # MODE B — local machine
    # Change these to your local paths
    BIGWIG_SOURCE = Path('C:/Users/YourName/PFE/bigwig')      # Windows example
    # BIGWIG_SOURCE = Path('/Users/YourName/PFE/bigwig')       # Mac example
    FINAL_H5      = Path('C:/Users/YourName/PFE/processed/epi_signals.h5')
    FINAL_MASK    = Path('C:/Users/YourName/PFE/processed/bin_mask.h5')
    WINDOW_PATH   = Path('C:/Users/YourName/PFE/processed/window_index.parquet')
    AVAIL_PATH    = Path('C:/Users/YourName/PFE/epigenetics/mark_availability.csv')
    CHECKPOINT    = Path('C:/Users/YourName/PFE/extraction_checkpoint.json')
    LOCAL_BIGWIG  = BIGWIG_SOURCE   # no staging needed for local mode
    LOCAL_H5      = FINAL_H5
    LOCAL_MASK    = FINAL_MASK

DRIVE_PROC.mkdir(parents=True, exist_ok=True)
LOCAL_BIGWIG.mkdir(parents=True, exist_ok=True)

In [ ]:
WINDOW_SIZE = 2048
N_BINS      = 128
BIN_SIZE    = 16
WRITE_BATCH = 512
HISTONE_MAX = 50.0
# ── Mark order — this is the canonical channel order for all downstream code
ALL_MARKS = [
    'H3K4me3',   # ch 0
    'H3K4me1',   # ch 1
    'H3K36me3',  # ch 2
    'H3K27me3',  # ch 3
    'H3K9me3',   # ch 4
    'H3K27ac',   # ch 5
    'H3K9ac',    # ch 6
    'DNase',     # ch 7
    'WGBS',      # ch 8
]
WGBS_MARK = 'WGBS'
N_MARKS   = len(ALL_MARKS)

EPIGENOMES = {
    'E003': 'H1_ESC',
    'E017': 'IMR90_fetal_lung',
    'E050': 'brain_angular_gyrus',
    'E067': 'brain_cortex',
    'E062': 'monocytes',
    'E045': 'HSC',
    'E065': 'aorta',
    'E071': 'brain_hippocampus',
    'E104': 'heart_right_atrium',
    'E116': 'GM12878_lymphoblastoid',
}


In [ ]:
# Checkpoint system — saves progress after every (epigenome × mark) pair
# ─────────────────────────────────────────────────────────────────────────────

def load_checkpoint():
    """
    Load the set of (eid, mark) pairs already completed.
    Returns an empty set if no checkpoint exists.
    """
    if CHECKPOINT.exists():
        data = json.loads(CHECKPOINT.read_text())
        completed = set(tuple(x) for x in data['completed'])
        print(f"Resuming from checkpoint: {len(completed)} pairs already done")
        return completed
    return set()


def save_checkpoint(completed: set):
    """Save the set of completed (eid, mark) pairs to disk."""
    data = {'completed': list(completed), 'timestamp': time.time()}
    CHECKPOINT.write_text(json.dumps(data))



In [ ]:
# Stage 1 (Colab only): Copy bigWig files from Drive to local /tmp
# ─────────────────────────────────────────────────────────────────────────────

def stage_bigwig_files(avail: pd.DataFrame):
    """
    Copy bigWig files from Drive to local /tmp.

    Why this matters:
    Drive FUSE latency = 20-80ms per random seek.
    Local NVMe latency = 0.1-0.5ms per random seek.
    This single step turns a 24-36 hour job into a 4-5 hour job.

    Copies only the files we'll actually use (avail=1), skips already-staged.
    A 700 MB bigWig copies in ~30s from Drive to /tmp over the internal
    Colab network — much faster than external download because it stays
    within Google's infrastructure.
    """
    if MODE != 'COLAB':
        return  # no staging needed for local mode

    pairs_to_stage = [
        (eid, mark)
        for eid  in EPIGENOMES
        for mark in ALL_MARKS
        if avail.loc[eid, mark] == 1
    ]

    already_staged = [
        (eid, mark) for eid, mark in pairs_to_stage
        if _local_bigwig_path(eid, mark).exists()
        and _local_bigwig_path(eid, mark).stat().st_size > 10_000_000
    ]

    to_copy = [p for p in pairs_to_stage if p not in already_staged]

    print(f"BigWig staging:")
    print(f"  Total needed  : {len(pairs_to_stage)}")
    print(f"  Already staged: {len(already_staged)}")
    print(f"  To copy now   : {len(to_copy)}")

    if not to_copy:
        print("  All files already staged ✅")
        return

    # Estimate copy time
    sample_size = _drive_bigwig_path(pairs_to_stage[0][0],
                                     pairs_to_stage[0][1]).stat().st_size
    avg_size_gb = sample_size / 1e9
    est_mins    = len(to_copy) * avg_size_gb * 60 / 3.0   # ~3 GB/min Drive→/tmp
    print(f"  Estimated copy time: {est_mins:.0f} minutes\n")

    for eid, mark in tqdm(to_copy, desc="Staging bigWig files"):
        src  = _drive_bigwig_path(eid, mark)
        dest = _local_bigwig_path(eid, mark)
        if not src.exists():
            print(f"  ⚠ Source missing: {src.name}")
            continue
        shutil.copy2(str(src), str(dest))

    staged_now = len([p for p in pairs_to_stage
                      if _local_bigwig_path(p[0], p[1]).exists()])
    total_gb   = sum(_local_bigwig_path(e, m).stat().st_size
                     for e, m in pairs_to_stage
                     if _local_bigwig_path(e, m).exists()) / 1e9
    print(f"\n  Staged: {staged_now}/{len(pairs_to_stage)} files  ({total_gb:.1f} GB)")
    print("  All bigWig files on local fast disk ✅")


In [ ]:
def _drive_bigwig_path(eid, mark):
    if mark == 'WGBS':
        return BIGWIG_SOURCE / f"{eid}_WGBS_FractionalMethylation.bigwig"
    return BIGWIG_SOURCE / f"{eid}-{mark}.pval.signal.bigwig"


def _local_bigwig_path(eid, mark):
    """Path on local fast disk (staged copy or original for local mode)."""
    if MODE == 'LOCAL':
        return _drive_bigwig_path(eid, mark)
    if mark == 'WGBS':
        return LOCAL_BIGWIG / f"{eid}_WGBS_FractionalMethylation.bigwig"
    return LOCAL_BIGWIG / f"{eid}-{mark}.pval.signal.bigwig"



In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Stage 2: Initialize HDF5 files (only runs on first call, skips if resuming)
# ─────────────────────────────────────────────────────────────────────────────

def initialize_hdf5_files(avail: pd.DataFrame, n_tiles: int):
    """
    Create HDF5 structure. If the file already exists (resume scenario),
    only add missing groups/datasets rather than overwriting.
    """
    for h5_path, dtype, fill in [
        (LOCAL_H5,   np.float16, 0.0),
        (LOCAL_MASK, np.uint8,   0),
    ]:
        mode = 'a' if h5_path.exists() else 'w'
        with h5py.File(h5_path, mode) as f:
            # Set file-level attrs only on first creation
            if 'n_tiles' not in f.attrs:
                f.attrs['n_tiles']       = n_tiles
                f.attrs['n_bins']        = N_BINS
                f.attrs['marks_order']   = ALL_MARKS
                f.attrs['epigenome_ids'] = list(EPIGENOMES.keys())

            for eid in EPIGENOMES:
                if eid not in f:
                    f.create_group(eid)
                grp = f[eid]
                for mark in ALL_MARKS:
                    if avail.loc[eid, mark] == 0:
                        continue
                    if mark not in grp:
                        grp.create_dataset(
                            mark,
                            shape    = (n_tiles, N_BINS),
                            dtype    = dtype,
                            chunks   = (min(WRITE_BATCH, n_tiles), N_BINS),
                            compression      = 'gzip',
                            compression_opts = 4,
                            fillvalue        = fill,
                        )

    print(f"HDF5 files ready (mode={'resume' if LOCAL_H5.exists() else 'new'})")


In [ ]:
# Stage 3: Extract one (epigenome × mark) pair — the core inner loop
# ─────────────────────────────────────────────────────────────────────────────

def extract_one_pair(
    eid:        str,
    mark:       str,
    tile_coords: list,   # pre-sorted list of (chrom, start, end)
    sig_f,               # open HDF5 signal file handle
    msk_f,               # open HDF5 mask file handle
) -> float:
    """
    Extract all tiles for one (epigenome × mark) pair.

    Returns elapsed seconds.

    Optimization notes:
    - tile_coords is pre-sorted by (chrom, start) so all queries on chr1
      come before chr2, etc. pyBigWig benefits from locality — the B-tree
      index stays warm in the OS page cache.
    - We open the bigWig file ONCE and keep it open for all 28K tiles.
      Opening a bigWig file takes ~50ms (parse chromosome header), so
      opening per-tile would add 28K × 50ms = 23 minutes of pure overhead.
    - Batch writes: accumulate 512 tiles in RAM (~512 KB float16), write once.
      HDF5 chunk flush overhead is ~1ms — doing it per-tile would add
      28K ms = 28 seconds of pure flush overhead per (eid, mark).
    """
    bw_path = _local_bigwig_path(eid, mark)
    if not bw_path.exists():
        print(f"  ⚠ bigWig not found: {bw_path.name}")
        return 0.0

    n_tiles = len(tile_coords)
    t_start = time.time()

    bw = pyBigWig.open(str(bw_path))

    # Validate chromosome naming convention once
    bw_chroms     = set(bw.chroms().keys())
    query_chroms  = set(c for c, _, _ in tile_coords)
    missing       = query_chroms - bw_chroms
    if missing:
        print(f"  ⚠ Chr mismatch in {bw_path.name}: {missing} not in bigWig")

    sig_ds = sig_f[eid][mark]
    msk_ds = msk_f[eid][mark]

    batch_sig  = np.zeros((WRITE_BATCH, N_BINS), dtype=np.float16)
    batch_msk  = np.zeros((WRITE_BATCH, N_BINS), dtype=np.uint8)
    batch_start = 0

    for tile_i, (chrom, start, end) in enumerate(tile_coords):
        # ── Query ──────────────────────────────────────────────────────────
        try:
            raw = bw.stats(chrom, start, end, type="mean", nBins=N_BINS)
        except RuntimeError:
            raw = [None] * N_BINS

        # ── Process None values ────────────────────────────────────────────
        msk = np.array([0 if v is None else 1 for v in raw], dtype=np.uint8)
        sig = np.array([0.0 if v is None else float(v) for v in raw],
                       dtype=np.float32)

        # ── Normalize ──────────────────────────────────────────────────────
        if mark == 'WGBS':
            sig = np.clip(sig, 0.0, 1.0)
        else:
            sig = np.clip(sig, 0.0, HISTONE_MAX)

        # ── Place in batch buffer ──────────────────────────────────────────
        local_i = tile_i - batch_start
        batch_sig[local_i] = sig.astype(np.float16)
        batch_msk[local_i] = msk

        # ── Flush when batch is full or last tile ──────────────────────────
        batch_end = tile_i + 1
        if local_i + 1 == WRITE_BATCH or batch_end == n_tiles:
            n = local_i + 1
            sig_ds[batch_start:batch_end] = batch_sig[:n]
            msk_ds[batch_start:batch_end] = batch_msk[:n]
            batch_start = batch_end
            batch_sig[:] = 0
            batch_msk[:] = 0

    bw.close()
    return time.time() - t_start



In [ ]:
# Stage 4: Copy final HDF5 from /tmp to Drive (Colab only)
# ─────────────────────────────────────────────────────────────────────────────

def copy_results_to_drive():
    """
    Copy the completed HDF5 files from /tmp to Drive.
    Only runs in COLAB mode. In LOCAL mode, results are written directly
    to the final destination.

    We write to /tmp throughout extraction because:
    - /tmp writes are ~10x faster than Drive FUSE writes
    - HDF5 with gzip compression does many small writes per chunk flush
    - Over Drive FUSE, each small write has 5-20ms overhead
    - Over 2.5M chunk flushes, that overhead would add hours

    The final copy is one large sequential write: fast even over FUSE.
    """
    if MODE != 'COLAB':
        return

    for src, dest in [(LOCAL_H5, FINAL_H5), (LOCAL_MASK, FINAL_MASK)]:
        if not src.exists():
            print(f"  ⚠ Source not found: {src}")
            continue
        size_gb = src.stat().st_size / 1e9
        print(f"Copying {src.name} ({size_gb:.2f} GB) → Drive...")
        t0 = time.time()
        shutil.copy2(str(src), str(dest))
        elapsed = time.time() - t0
        print(f"  Done in {elapsed/60:.1f} min  ✅  {dest}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Main orchestration
# ─────────────────────────────────────────────────────────────────────────────

def run():
    print("=" * 60)
    print(f"SIGNAL EXTRACTION  (mode={MODE})")
    print("=" * 60)

    # ── Load inputs ───────────────────────────────────────────────────────
    print(f"\nLoading window coordinates from {WINDOW_PATH}...")
    window_df   = pd.read_parquet(WINDOW_PATH)
    n_tiles     = len(window_df)
    print(f"  Tiles: {n_tiles:,}")

    avail = pd.read_csv(AVAIL_PATH, index_col=0).astype(np.uint8)
    total_pairs = int(avail.values.sum())
    print(f"  Available (eid, mark) pairs: {total_pairs}")

    # ── Sort tiles by chromosome then position ────────────────────────────
    # This is the single most impactful optimization for pyBigWig performance.
    # bigWig files are indexed B-trees — sequential genomic queries stay in
    # the same index branch and benefit from OS page cache.
    chrom_order  = {f'chr{i}': i for i in range(1, 23)}
    chrom_order.update({'chrX': 23, 'chrY': 24, 'chrM': 25})
    window_df    = window_df.copy()
    window_df['_chrom_ord'] = window_df['chrom'].map(
        lambda c: chrom_order.get(c, 99)
    )
    window_df    = window_df.sort_values(['_chrom_ord', 'win_start'])
    tile_coords  = list(zip(
        window_df['chrom'].values,
        window_df['win_start'].astype(int).values,
        window_df['win_end'].astype(int).values,
    ))
    # Keep a mapping from sorted position → original HDF5 row index
    # We need this so data lands in the right row in the HDF5 datasets
    # (HDF5 row i must correspond to window_df.iloc[i] in unsorted order)
    sorted_to_original = window_df.index.tolist()
    print(f"  Tiles sorted by genomic position ✅")

    # ── Load checkpoint ───────────────────────────────────────────────────
    completed   = load_checkpoint()
    remaining   = [
        (eid, mark)
        for eid  in EPIGENOMES
        for mark in ALL_MARKS
        if avail.loc[eid, mark] == 1 and (eid, mark) not in completed
    ]
    print(f"\n  Pairs completed : {len(completed)}/{total_pairs}")
    print(f"  Pairs remaining : {len(remaining)}")

    if not remaining:
        print("\n✅ All pairs already extracted. Copying to Drive if needed.")
        copy_results_to_drive()
        return

    # ── Stage bigWig files ────────────────────────────────────────────────
    stage_bigwig_files(avail)

    # ── Initialize HDF5 ───────────────────────────────────────────────────
    initialize_hdf5_files(avail, n_tiles)

    # ── Runtime estimate ──────────────────────────────────────────────────
    ms_per_tile = 0.4 if MODE == 'LOCAL' else 0.5  # local SSD after staging
    est_hours   = len(remaining) * n_tiles * ms_per_tile / 1000 / 3600
    print(f"\n  Estimated remaining time: {est_hours:.1f} hours")
    print(f"  Checkpointing after each of the {len(remaining)} remaining pairs\n")

    # ── Main extraction loop ──────────────────────────────────────────────
    total_elapsed = 0.0
    pairs_done    = 0

    with h5py.File(LOCAL_H5,   'a') as sig_f, \
         h5py.File(LOCAL_MASK, 'a') as msk_f:

        for eid, mark in remaining:
            print(f"  [{pairs_done+1}/{len(remaining)}] {eid} — {mark:<12}", end="", flush=True)

            elapsed = extract_one_pair(
                eid, mark, tile_coords, sig_f, msk_f
            )
            total_elapsed += elapsed
            pairs_done    += 1

            # Speed stats
            ms_per_tile_actual = elapsed / n_tiles * 1000
            remaining_pairs    = len(remaining) - pairs_done
            eta_hours          = remaining_pairs * elapsed / 3600
            print(f"  {elapsed:.0f}s  ({ms_per_tile_actual:.2f} ms/tile)  "
                  f"ETA: {eta_hours:.1f}h")

            # Checkpoint — save progress so restart resumes here
            completed.add((eid, mark))
            save_checkpoint(completed)

    print(f"\nExtraction complete in {total_elapsed/3600:.1f} hours total")

    # ── Copy to Drive ─────────────────────────────────────────────────────
    print("\nCopying results to Drive...")
    copy_results_to_drive()

    # ── Build mark mask matrix ────────────────────────────────────────────
    mark_mask_path = DRIVE_PROC / 'mark_mask_matrix.npy'
    matrix = avail[ALL_MARKS].values.astype(np.uint8)
    np.save(mark_mask_path, matrix)
    print(f"Saved mark_mask_matrix.npy: {mark_mask_path}")

    print("\n✅ All done. Proceed to verification (02b_verification.py)")


if __name__ == '__main__':
    run()

SIGNAL EXTRACTION  (mode=COLAB)

Loading window coordinates from /content/drive/MyDrive/PFE/data/processed/window_index.parquet...
  Tiles: 29,790
  Available (eid, mark) pairs: 75
  Tiles sorted by genomic position ✅

  Pairs completed : 0/75
  Pairs remaining : 75
BigWig staging:
  Total needed  : 75
  Already staged: 0
  To copy now   : 75
  Estimated copy time: 943 minutes



Staging bigWig files: 100%|██████████| 75/75 [11:34<00:00,  9.27s/it]


  Staged: 75/75 files  (44.6 GB)
  All bigWig files on local fast disk ✅
HDF5 files ready (mode=resume)

  Estimated remaining time: 0.3 hours
  Checkpointing after each of the 75 remaining pairs

  [1/75] E003 — H3K4me3     

  289s  (9.69 ms/tile)  ETA: 5.9h
  [2/75] E003 — H3K4me1       294s  (9.88 ms/tile)  ETA: 6.0h
  [3/75] E003 — H3K36me3      288s  (9.67 ms/tile)  ETA: 5.8h
  [4/75] E003 — H3K27me3      297s  (9.98 ms/tile)  ETA: 5.9h
  [5/75] E003 — H3K9me3       288s  (9.67 ms/tile)  ETA: 5.6h
  [6/75] E003 — H3K27ac       293s  (9.84 ms/tile)  ETA: 5.6h
  [7/75] E003 — H3K9ac        304s  (10.21 ms/tile)  ETA: 5.7h
  [8/75] E003 — DNase         288s  (9.66 ms/tile)  ETA: 5.4h
  [9/75] E003 — WGBS          267s  (8.95 ms/tile)  ETA: 4.9h
  [10/75] E017 — H3K4me3       286s  (9.60 ms/tile)  ETA: 5.2h
  [11/75] E017 — H3K4me1       295s  (9.90 ms/tile)  ETA: 5.2h
  [12/75] E017 — H3K36me3      292s  (9.79 ms/tile)  ETA: 5.1h
  [13/75] E017 — H3K27me3      294s  (9.87 ms/tile)  ETA: 5.1h
  [14/75] E017 — H3K9me3       288s  (9.67 ms/tile)  ETA: 4.9h
  [15/75] E017 — H3K27ac       295s  (9.91 ms/tile)  ETA: 4.9h
  [16/75] E017 — H3K9ac        291s  (9.77 ms/tile)  ETA: 4.8h
  [17/75] E017 — DNase      

In [ ]:
# Biological sanity checks
# ─────────────────────────────────────────────────────────────────────────────

def run_sanity_checks(window_df: pd.DataFrame, avail: pd.DataFrame):
    """
    Verify the extracted signals are biologically sensible.

    Checks performed:
    1. No NaN or Inf in any signal
    2. WGBS values are in [0, 1]
    3. Histone values are in [0, 50]
    4. Signal is non-zero for at least some tiles per mark
    5. Mark-level statistics match biological expectations:
       - H3K4me3 should be enriched at TSS windows
       - H3K27me3 should have low mean and distinct high-value tail
       - WGBS at CpG island windows should show bimodal distribution
         (active promoters ~0.1, inactive regions ~0.7–0.9)
    """
    print("\n" + "═" * 60)
    print("BIOLOGICAL SANITY CHECKS")
    print("═" * 60)

    eids = [eid for eid in EPIGENOMES.keys()
            if any(avail.loc[eid, m] == 1 for m in ALL_MARKS)]

    with h5py.File(EPI_H5_PATH, 'r') as f:

        for eid in eids[:3]:   # check first 3 epigenomes to keep runtime short
            print(f"\n── {eid} ({EPIGENOMES[eid]}) ──")

            for mark in ALL_MARKS:
                if avail.loc[eid, mark] == 0:
                    print(f"  {mark:<12}: not available")
                    continue

                sig = f[eid][mark][:].astype(np.float32)  # (N, 128)
                flat = sig.flatten()

                # Check 1: no NaN/Inf
                assert not np.isnan(flat).any(), \
                    f"NaN found in {eid}/{mark}!"
                assert not np.isinf(flat).any(), \
                    f"Inf found in {eid}/{mark}!"

                # Check 2: range
                if mark == WGBS_MARK:
                    assert flat.min() >= -0.001 and flat.max() <= 1.001, \
                        f"WGBS out of [0,1]: min={flat.min():.3f} max={flat.max():.3f}"
                else:
                    assert flat.min() >= -0.001 and flat.max() <= 50.001, \
                        f"{mark} out of [0,50]: max={flat.max():.3f}"

                # Check 3: not all zeros (would indicate extraction failure)
                mean_sig = flat.mean()
                frac_nonzero = (flat > 0.01).mean()
                assert frac_nonzero > 0.01, \
                    f"{eid}/{mark} is almost entirely zero! Extraction likely failed."

                # Summary statistics
                print(f"  {mark:<12}: mean={mean_sig:.3f}  "
                      f"max={flat.max():.2f}  "
                      f"nonzero={frac_nonzero*100:.1f}%  ✅")

    # ── Cross-mark biological check ───────────────────────────────────────
    # For E003 (H1 ESC, has all marks), check:
    # H3K4me3 should be the most enriched mark at CpG island windows
    # because these windows were selected to contain CpG islands,
    # and ~70% of CpG islands overlap TSSs which carry H3K4me3.
    print("\n── Cross-mark enrichment ranking (E003, expected: K4me3 highest) ──")
    with h5py.File(EPI_H5_PATH, 'r') as f:
        if 'E003' in f:
            means = {}
            for mark in ALL_MARKS:
                if mark in f['E003']:
                    means[mark] = f['E003'][mark][:].mean()
            ranked = sorted(means.items(), key=lambda x: -x[1])
            for rank, (mark, mean_val) in enumerate(ranked, 1):
                expected_note = ""
                if mark == 'H3K4me3' and rank == 1:
                    expected_note = "  ← expected highest"
                elif mark in ('H3K27me3', 'H3K9me3') and rank > 5:
                    expected_note = "  ← repressive marks, expected lower"
                print(f"  {rank}. {mark:<12}: {mean_val:.4f}{expected_note}")

    print("\n✅ All sanity checks passed")
